# 2 · Wind farm layout
### المصدر · Source — رحلة الكم من مصدر الطاقة إلى الاستدامة · Quantum Journey from Energy Source to Sustainability
*Quantum Computing Challenge 2026, Riyadh · Team 2 · Prepared by Ahmed Elghadban, Quantumy*<br><sub>Reviewed by Ola Tarig</sub>

| | |
|---|---|
| Use case | Choose which 3 of 6 candidate sites get a turbine so that yield minus wake losses is best |
| Algorithm | VQE (Variational Quantum Eigensolver) on the placement QUBO |
| Toy size | 6 qubits, runs in under 3 min on a laptop |
| Day 1 | read sections 0–2 |
| Day 2 | sections 3–6, build, pitch |

In [ ]:
# Setup: run me first. On Colab this clones the repo and installs the pinned packages (about 1 minute).
import sys, os, subprocess
if "google.colab" in sys.modules and not os.path.exists("Quantum-Coding-Challenge"):
    subprocess.run(["git", "clone", "-q", "https://github.com/QTACInstitute/Quantum-Coding-Challenge.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "Quantum-Coding-Challenge/requirements.txt"], check=True)
ROOT = "Quantum-Coding-Challenge" if os.path.exists("Quantum-Coding-Challenge/qff") else ".."
sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, matplotlib.pyplot as plt
import qff
SEED = qff.SEED            # 2026
DATA = os.path.join(ROOT, "data")

## 0 · The journey so far — رحلة الكم
**«من المصدر إلى استخدام الطاقة»** · From the wellspring to the lamp

Before you switch on a lamp tonight, energy will have made one journey. It is found, it is moved, it is stored, and then it is used. Ten teams follow that journey today. Each team owns one moment of it, and each moment has a problem that is hard for ordinary computers at full scale.

1. **المصدر · Source.** Find the energy and decide where to harvest it: read the rock under the ground, place the turbines in the wind.
2. **الطريق · Move.** Get it across the country: watch the grid, route the ships, decide which power stations run.
3. **المخزن · Store.** Keep it for later, and keep carbon out of the air: batteries, hydrogen, capture materials. This is chemistry.
4. **استخدام الطاقة · Use.** Spend it well: make fresh water, charge and discharge a fleet of cars.

Your notebook is one chapter. The cell below lights your stop and lists all ten chapters so you can see who comes before and after you. When you pitch, start by saying where in the journey you are.

One rule for the whole challenge: **be honest about what is real.** As of November 2026 nobody has shown a quantum computer beating the best classical method on any of these ten problems. Your job is to explain the problem, show the idea on a small example, and say clearly what the small example does and does not prove.

In [ ]:
from qff.day import show_day
show_day(stop=1, challenge=2)

## 1 · This stop — هذه المحطة

**The problem.** A wind farm has a handful of good spots for turbines. Each turbine takes energy out of the wind, so the air behind it is slower and more turbulent: a turbine standing in that *wake* produces less. Choosing which spots to build on is a trade-off between the yield of each spot and the wake losses between spots. Here the question is: *build exactly K turbines, on which sites, so that total yield minus wake losses is as large as possible?*

**Who faces it.** A wind-farm developer, or an energy company deciding where to put turbines on a leased site.

<div style="border:1px solid rgba(208,38,112,0.35);border-left:6px solid #D02670;background:rgba(208,38,112,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#D02670;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🌍 Real world · في الواقع</span>

**What is real today.** The WEF/Aramco paper (Case 8, p. 20) describes a QUBO formulation of wind-farm layout solved with a hybrid VQE on IBM's Qiskit gate-based simulator, and says this is viable "for small instances". Classical layout optimisers (mathematical programming and heuristic search) are mature and handle far larger farms, with continuous positions and real wake physics. As far as we know, nobody has shown the quantum route beating them (see the rule in section 0).

</div>

**Why a quantum computer might help.** The number of layouts doubles with every candidate site, and VQE looks for the lowest-energy state of a quantum system whose energies are the layout costs. What is still missing: the search is a noisy, trial-and-error loop that often stops in the wrong valley, and no one has shown it beats the classical optimisers at a useful size.

| Paper's method | Our teaching route |
|---|---|
| QUBO formulation, hybrid VQE on IBM's Qiskit gate-based simulator, "small instances" (Case 8, p. 20) | The same route: a 6-site QUBO and a VQE run on a Qiskit simulator. Our yields and wake penalties are invented for teaching |

## 2 · Made small — المسألة مصغَّرة

Six candidate sites on a 2×3 grid, with the wind blowing **from the west** (left to right). We number the sites `3·row + col`, so the top row is 0, 1, 2 and the bottom row is 3, 4, 5. We must build **exactly K = 3** turbines.

A **layout** is a list of six 0/1 values `x = (x0, ..., x5)`: `x_i = 1` means "build a turbine at site i". Six yes/no questions give 2⁶ = 64 layouts, so six qubits hold one amplitude per layout.

Each site has a **yield** (what a turbine there would produce alone). Each pair of sites has a **wake penalty**: if both get a turbine, the one behind loses some output. Neighbours along the wind lose most (4), the next one downwind loses less (2), diagonal neighbours lose a little (1). The numbers are invented for teaching.

The cost to **minimise** is `−Σ yield_i·x_i + Σ wake_ij·x_i·x_j`, subject to `Σ x_i = 3`.

In [ ]:
QUBITS = 6                       # the test runner reads this
import time
from qiskit_optimization import QuadraticProgram

K = 3
yields = [10, 10, 10, 10, 9, 8]
wake = {(0, 1): 4, (1, 2): 4, (0, 2): 2, (3, 4): 4, (4, 5): 4, (3, 5): 2, (0, 4): 1, (1, 5): 1, (3, 1): 1, (4, 2): 1}
POS = {i: (i % 3, -(i // 3)) for i in range(6)}          # column to the right, row downwards


def draw_farm(chosen=(), title="", ax=None):
    """Draw the 2x3 site grid. Sites in `chosen` get a turbine (blue); wake links between two turbines are red, others grey."""
    ax = ax or plt.subplots(figsize=(4.6, 2.6))[1]
    for (i, j), w in wake.items():
        both = i in chosen and j in chosen
        ax.plot([POS[i][0], POS[j][0]], [POS[i][1], POS[j][1]], lw=1 + w * (1.2 if both else 0.4),
                color="#FA4D56" if both else "#E0E0E0", zorder=1)
    for i in range(6):
        on = i in chosen
        ax.scatter(*POS[i], s=900, color="#0F62FE" if on else "#F4F4F4", edgecolor="#161616", zorder=2)
        ax.text(POS[i][0], POS[i][1], f"{i}\n{yields[i]}", ha="center", va="center", fontsize=9, color="white" if on else "black", zorder=3)
    ax.annotate("wind", xy=(2.55, 0.55), xytext=(-0.55, 0.55), arrowprops=dict(arrowstyle="->", color="#697077"), color="#697077", va="center")
    ax.set_xlim(-0.7, 2.7); ax.set_ylim(-1.5, 0.9)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
    return ax


# The problem as Qiskit sees it: a QuadraticProgram with a real constraint.
qp = QuadraticProgram("wind_farm")
for i in range(6):
    qp.binary_var(f"x{i}")
qp.minimize(linear={f"x{i}": -yields[i] for i in range(6)},
            quadratic={(f"x{i}", f"x{j}"): w for (i, j), w in wake.items()})
qp.linear_constraint(linear={f"x{i}": 1 for i in range(6)}, sense="==", rhs=K, name="exactly_K")
print(qp.prettyprint())
draw_farm(title="six sites (number, yield); wind from the west");

**Classical baseline first.** Only 64 layouts exist, so we just try them all and keep the cheapest one that builds exactly 3 turbines. `qff.classical.brute_force` does that. We also ask Qiskit's classical exact solver (`NumPyMinimumEigensolver`) to solve the same problem as a cross-check.

In [ ]:
# Classical baseline: always runs first.
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_algorithms.utils import algorithm_globals
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qff.bits import index_to_x, to_x

algorithm_globals.random_seed = SEED

t0 = time.perf_counter()
classical_x, classical_value, table = qff.classical.brute_force(qp)
brute_seconds = time.perf_counter() - t0
feasible_rows = [r for r in table if r[2]]
print(f"layouts checked: {len(table)} | feasible (exactly {K} turbines): {len(feasible_rows)}")
print("best layout x:", classical_x, "= sites", [i for i in range(6) if classical_x[i]], "| objective:", classical_value)
print("next best feasible values:", sorted({r[1] for r in feasible_rows})[1:4], "| how many layouts tie at each:",
      [sum(1 for r in feasible_rows if r[1] == v) for v in sorted({r[1] for r in feasible_rows})[:4]])

t0 = time.perf_counter()
numpy_result = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)
numpy_seconds = time.perf_counter() - t0
print("exact classical solver:", [int(v) for v in numpy_result.x], numpy_result.fval)

fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
draw_farm([0, 1, 2], "{0, 1, 2}: all in the front row", ax=axes[0])
draw_farm([i for i in range(6) if classical_x[i]], f"best layout, objective {classical_value:.0f}", ax=axes[1])
plt.tight_layout()

Brute force answers in a millisecond. That is the honest starting point for everything below: **on this toy the classical method is instant, and VQE cannot beat instant.** The toy exists so you can see how the quantum method works, and where it struggles.

### 🧰 Toolbox: how the calls you will need are written · صندوق الأدوات

No internet? No problem. The cell under this table runs each call once on a **small toy that has nothing to do with the task**, so you can see the exact spelling, the order of the arguments, and what comes back. Run it, read the printed lines, and come back to it whenever a hint says "a method of ...". *What* to build, and with *which numbers*, is your job.

| To do | How it is written | What you get |
|---|---|---|
| a circuit with knobs | `ansatz = real_amplitudes(n, reps=1, entanglement="linear")` | `ansatz.num_parameters` tells how many knobs |
| the optimiser | `COBYLA(maxiter=400)` | an object that turns the knobs |
| one VQE run | `VQE(estimator, ansatz, optimizer, initial_point=x0, callback=fn)` | an object; nothing runs yet |
| run it on an operator | `res = vqe.compute_minimum_eigenvalue(op)` | `res.eigenvalue.real` (lowest energy found, **without** any constant offset), `res.optimal_point` (best knobs) |
| the callback | `def fn(eval_count, params, energy, metadata): ...` | called by VQE after every energy evaluation; keep the energy |
| probabilities of a state | `Statevector(ansatz.assign_parameters(knobs)).probabilities_dict()` | `{'010110': 0.52, ...}`; the **last** character is qubit 0 |
| the most likely key | `max(probs, key=probs.get)` | the string with the largest probability |
| string to layout | `qff.bits.to_x('110')` | `[0, 1, 1]`: the list $x$ with $x_0$ first |
| seeded random start | `np.random.default_rng(seed).uniform(-np.pi, np.pi, n)` | $n$ numbers between $-\pi$ and $\pi$, the same on every run with the same seed |

In [ ]:
# Toolbox for 02: one complete VQE call on a 1-qubit toy (find the lowest energy of the operator Z; the answer is -1).
from qiskit.circuit.library import real_amplitudes
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA

toy_op = SparsePauliOp("Z")                               # the energy operator
toy_ansatz = real_amplitudes(1, reps=1)                   # a circuit with knobs
trace = []


def record(eval_count, params, energy, metadata):          # VQE calls this after every energy evaluation, with these four arguments
    trace.append(energy)


vqe_toy = VQE(StatevectorEstimator(), toy_ansatz, COBYLA(maxiter=100), initial_point=[0.3] * toy_ansatz.num_parameters, callback=record)
res = vqe_toy.compute_minimum_eigenvalue(toy_op)
print("lowest energy found:", round(res.eigenvalue.real, 4), "| best knobs:", np.round(res.optimal_point, 3), "| energies recorded:", len(trace))
probs = Statevector(toy_ansatz.assign_parameters(res.optimal_point)).probabilities_dict()
print("probabilities of the final state:", {str(k): round(float(v), 3) for k, v in probs.items()}, "| most likely:", max(probs, key=probs.get))
print("string to list:", qff.bits.to_x("110"), "(x_0 first)")
print("seeded random start (3 numbers, seed 1):", np.round(np.random.default_rng(1).uniform(-np.pi, np.pi, 3), 3))
assert abs(res.eigenvalue.real + 1) < 1e-3 and qff.bits.to_x("110") == [0, 1, 1]
print("OK: toolbox 02 ran.")

### T1 · Warm-up: score a layout by hand · التمرين ١

**What you are about to build, and why.** Everything in this notebook (the penalty, the Hamiltonian, the quantum circuit) is built to *rank layouts*. Before a quantum computer can rank them, you must be able to score one yourself. In T1 you write `layout_value(sites)`, which returns the objective of a layout, and you use it to see *why* the front row is a poor choice even though its yield is the same.

**The formula you implement.** For a layout $x$ (with $x_i=1$ when site $i$ gets a turbine), yields $y_i$ and wake penalties $w_{ij}$ (pairs in the dictionary `wake`):

$$f(x)=-\sum_i y_i\,x_i\;+\sum_{(i,j)\in W}w_{ij}\,x_i\,x_j \qquad\text{subject to}\qquad \sum_i x_i=K \tag{1}$$

The product $x_ix_j$ is 1 only if **both** sites are chosen, so a wake penalty is paid only for pairs that are both built. The minus sign on the yield means "lower is better": we *minimise* $f$.

**Stuck? Open the hints one at a time, and try again after each. · تلميحات**

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 1 · a nudge · تلميح ١</summary>

A layout is a list of site numbers, for example `[0, 2, 3]`. Two things must be added up: what you *gain* (the yield of every chosen site) and what you *lose* (the wake penalty of every pair in which **both** sites are chosen). Look at the `wake` dictionary in the cell above: how are its keys written? Try the idea on a layout with a single site first: with no pair of sites, which of the two parts must come out as zero?

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 2 · the structure · تلميح ٢</summary>

Three parts: the total yield of the chosen sites; the total penalty over the entries of `wake` whose two sites are both in the layout; and the objective, which is the loss minus the gain (we minimise). Then call the function for the two layouts and compare their yields and their wake losses in one sentence. Two practical points: each key of `wake` is a pair of site numbers, so ask whether *both* numbers of a key appear in `sites`; and remember the sign, because the function returns a cost, so a layout with more yield must get a more negative value.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 3 · nearly there · تلميح ٣</summary>

```python
def layout_value(sites):
    gain = ______       # total yield of the chosen sites
    loss = ______       # total wake penalty of the pairs in `wake` whose two sites are both chosen
    return ______       # equation (1)
v_front = ______        # the front row
v_good = ______         # sites 0, 2 and 3
why = "..."             # same yield, but what differs?
```
Test yourself: the two values must be -20 and -28.

</details>

In [ ]:
# TODO (team) T1: compute the objective of {0,1,2} and {0,2,3} by hand and explain the difference
# Goal: layout_value(sites) returns the objective (equation 1) of a layout given as a list of site numbers.
# Keep v_front (layout [0, 1, 2]) and v_good (layout [0, 2, 3]), then write why = "..." in one sentence.
raise NotImplementedError("T1: write layout_value, evaluate {0,1,2} and {0,2,3}, explain the difference")

In [ ]:
# Self-check
assert v_front == -20, "T1: {0,1,2} should give -30 + 10 = -20"
assert v_good == -28, "T1: {0,2,3} should give -30 + 2 = -28"
assert layout_value([i for i in range(6) if classical_x[i]]) == classical_value, "T1: layout_value should agree with brute force on the best layout"
assert isinstance(why, str) and len(why) > 30, "T1: write your explanation in a sentence"

## 3 · The algorithm — الخوارزمية

### 3.0 · The map of this section

<div style="border:1px solid rgba(0,125,121,0.35);border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">💡 Idea · فكرة</span>

**VQE in one sentence.** Write "how good is this layout" as an *energy*, prepare a quantum state with a circuit whose knobs we can turn, measure the average energy of that state, and let a classical optimiser turn the knobs until the energy is as low as it can get: the lowest energy belongs to the best layout.

</div>

Everything below builds that sentence from two things you already know: lists of numbers (vectors) and probability. Every step has a **numbered equation** and, right after it, the **code cell that implements it**, so you can watch the maths run and check it against Qiskit.

| Step | What happens | Equation | Where in the code | Your task |
|---|---|---|---|---|
| (section 2) | The objective of a layout | (1) | `layout_value` | **T1** |
| 3.1 | The rule "exactly K" becomes a penalty: one cost for every layout | (2) | `QuadraticProgramToQubo` | |
| 3.2 | The cost becomes a Hamiltonian (an energy) | (3) | `qubo.to_ising()` | |
| 3.3 | Why lowering the average energy finds the best layout | (4), (5) | energy check cell | |
| 3.4 | The tunable circuit (the ansatz) and its knobs | (6), (7) | `real_amplitudes` | |
| 3.5 | Measuring the average energy | (4) again | `energy_at` | |
| 3.6 | The VQE loop and the restarts | (8) | `run_vqe` | **T2** |
| 3.7 | Deeper circuits, and the penalty | (2) again | `reps=2`, `penalty=2` | **T3** |

Words in **bold** are new ideas; each is explained where it first appears, and the Arabic term sits in a small table next to it.

In [ ]:
qff.plots.plot_flow(
    [("Layout cost", "equations 1-2", "source"), ("Hamiltonian", "equation 3", "move"), ("Ansatz circuit", "equations 6-7", "store"),
     ("Average energy", "equations 4-5", "use")],
    loop=(3, 2, "the classical optimiser turns the knobs (equation 8)"), title="VQE: a quantum circuit and a classical optimiser, in a loop");

**Symbol table** (keep it open while you read; the right column is where the symbol lives in the code).

| Symbol | Meaning | In the code |
|---|---|---|
| $x=(x_0,\dots,x_5)$ | a layout: turbine at site $i$ when $x_i=1$ | `x`, `index_to_x(k, 6)` |
| $y_i,\ w_{ij},\ K$ | yield of site $i$, wake penalty of a pair, number of turbines | `yields`, `wake`, `K` |
| $f(x),\ P,\ C(x)$ | objective, penalty weight, cost with penalty | `layout_value`, `PENALTY`, `diag[k]` |
| $Z_i$ | Pauli $Z$ on qubit $i$ | the Pauli strings of `cost_op` |
| $H,\ h_i,\ J_{ij},\ c_0'$ | cost Hamiltonian and its coefficients | `cost_op`, `cost_op.coeffs`, `offset` |
| $\alpha_x,\ p(x)$ | amplitude and probability of layout $x$ | `Statevector(...).data`, `.probabilities()` |
| $\boldsymbol\theta$ | the 12 knobs of the ansatz | `ansatz.parameters`, `x0`, `result.optimal_point` |
| $\langle H\rangle(\boldsymbol\theta),\ E_0$ | average energy of the circuit; lowest cost | `energy_at(theta)`, `diag.min()` |

<div style="border:1px solid rgba(15,98,254,0.35);border-left:6px solid #0F62FE;background:rgba(15,98,254,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#0F62FE;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">∑ Math · رياضيات</span>&nbsp; **The line to remember**

**The one line to remember.** For *any* state, $\langle H\rangle=\sum_xp(x)\,C(x)\ \ge\ C_{\min}$ (equation 5). VQE only ever has to push the average energy down; the floor is the best layout.

</div>

### 3.1 · From a rule to a cost: the penalty

A **layout** is a list of six yes/no decisions $x=(x_0,\dots,x_5)$, so there are $2^6=64$ of them. Equation (1) scores a layout, but it does not yet know about the rule "build **exactly $K=3$** turbines": a layout with four turbines could score better than any legal one. We fold the rule into the cost as a **penalty**:

$$C(x)=f(x)+P\Bigl(\sum_i x_i-K\Bigr)^2 \tag{2}$$

The bracket is 0 when the number of turbines is right and positive otherwise, so the penalty is paid only by illegal layouts (layouts that obey the rule are called *legal* or *feasible*); $P>0$ says how much. Because a 0/1 variable satisfies $x_i^2=x_i$, the square expands into single and pair terms:

$$P\Bigl(\sum_i x_i-K\Bigr)^2=PK^2+P(1-2K)\sum_i x_i+2P\sum_{i<j}x_ix_j$$

so $C$ is again a sum of single-site terms and pair terms: a **QUBO** (*quadratic unconstrained binary optimisation*: quadratic means "at most pairs", unconstrained means "the rule is inside the cost", binary means 0/1 variables). We use $P=12$, larger than any single yield, so that breaking the rule never pays (T3 shows what happens with a small $P$). Qiskit does the folding for us with `QuadraticProgramToQubo(penalty=12)`; the cell after the next one checks it against equation (2).

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**Penalty too small = wrong problem.** If $P$ is small, an illegal layout can have a *lower* cost than every legal one, and then even a perfect optimiser returns an illegal layout. T3 lets you see this.

</div>

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Penalty | العقوبة (penalty) | I | QUBO | مسألة تحسين ثنائية تربيعية غير مقيدة (QUBO) | I |
| Constraint | القيد | I | Layout | التخطيط (توزيع المواقع) | I |
| Cost function | دالة التكلفة | I | Wake | أثر التعاقب (الدوامة الخلفية) | I |
| Optimization | الأمثَلة (التحسين) | I | Wind turbine | توربين رياح | I |

In [ ]:
from qiskit.circuit.library import real_amplitudes
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.converters import QuadraticProgramToQubo

# Steps 1 and 2: constraint -> penalty -> Ising Hamiltonian. penalty=12 is larger than any single yield, so breaking the rule never pays.
qubo = QuadraticProgramToQubo(penalty=12).convert(qp)
cost_op, offset = qubo.to_ising()
diag = qff.classical.diag_of(cost_op, offset)              # the cost of every one of the 64 states, as an array
print(f"qubits: {cost_op.num_qubits} | Pauli terms: {len(cost_op)} | constant offset: {offset}")
print("lowest cost on the diagonal:", diag.min(), "| average over all 64 layouts:", round(diag.mean(), 2))
print("first terms (Pauli strings are little-endian: the rightmost letter is qubit 0):")
for pauli, coeff in list(zip(cost_op.paulis, cost_op.coeffs))[:4]:
    print("  ", pauli, coeff.real)

**Check equation (2) against Qiskit.** The cell below recomputes the cost of all 64 layouts from equations (1) and (2), using *your* `layout_value` from T1, and stops with an error if it disagrees with what Qiskit's converter produced.

In [ ]:
# Check: equation (2) reproduces Qiskit's cost table (the diagonal of the Hamiltonian) for all 64 layouts.
x_all = [index_to_x(k, 6) for k in range(64)]
PENALTY = 12


def cost_by_formula(x, P=PENALTY):
    """Equation (2): f(x) from your layout_value (T1) plus P * (number of turbines - K)^2."""
    return layout_value([i for i in range(6) if x[i]]) + P * (sum(x) - K) ** 2


assert np.allclose(diag, [cost_by_formula(x) for x in x_all]), "equation (2) disagrees with Qiskit's cost table"
print("OK: equation (2) matches Qiskit for all 64 layouts.")
legal = [k for k, x in enumerate(x_all) if sum(x) == K]
print(f"legal layouts (exactly {K} turbines): {len(legal)} of 64 | cheapest legal cost {diag[legal].min()} | cheapest illegal cost {np.delete(diag, legal).min()}")

### 3.2 · From a QUBO to a Hamiltonian

A quantum computer works with **operators** that act on the state. An operator is a *matrix*: a square table of numbers that turns one state vector into another. We need a matrix whose "energy" for layout $x$ is exactly $C(x)$. Physicists call it a **Hamiltonian** (an energy operator).

The **Pauli $Z$ gate** is the matrix $Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix}$: it leaves $|0\rangle$ alone and flips the sign of $|1\rangle$, so $Z|0\rangle=(+1)|0\rangle$ and $Z|1\rangle=(-1)|1\rangle$. We say $|0\rangle$ and $|1\rangle$ are **eigenstates** of $Z$ with **eigenvalues** $+1$ and $-1$. Notice that $\frac{1-Z}{2}$ has eigenvalue $0$ on $|0\rangle$ and $1$ on $|1\rangle$: exactly the value of a bit. So replace every variable of the QUBO by an operator, $x_i\to\frac{1-Z_i}{2}$ and $x_ix_j\to\frac{(1-Z_i)(1-Z_j)}{4}$, and multiply out. For a QUBO $C=c_0+\sum_i l_i x_i+\sum_{i<j}q_{ij}x_ix_j$ the result is

$$\begin{aligned}H&=c_0'+\sum_i h_i\,Z_i+\sum_{i<j}J_{ij}\,Z_iZ_j,\\ c_0'&=c_0+\sum_i\frac{l_i}{2}+\sum_{i<j}\frac{q_{ij}}{4},\quad h_i=-\frac{l_i}{2}-\sum_{j\ne i}\frac{q_{ij}}{4},\quad J_{ij}=\frac{q_{ij}}{4}\end{aligned} \tag{3}$$

(where $q_{ij}$ with $j<i$ means $q_{ji}$). A Hamiltonian made only of $Z$ terms is called an **Ising** Hamiltonian (that is why the Qiskit method is called `to_ising`). It is a **diagonal** matrix (every number off the diagonal is 0) with the costs on its diagonal: $H|x\rangle=C(x)\,|x\rangle$. The best layout is the state of lowest energy, the **ground state**. **Finding the best layout means finding the ground state of $H$.** Qiskit's Hamiltonian leaves out the constant $c_0'$ (it only shifts every energy by the same amount), and `to_ising()` returns it separately as `offset`: **add it back whenever you compare with a cost.**

**A tiny example you can check on paper.** Two sites, yields $3$ and $2$, exactly $K=1$ turbine, penalty $P=5$. By (2): $C(x_0,x_1)=-3x_0-2x_1+5(x_0+x_1-1)^2$.

| layout $(x_0,x_1)$ | yield part | $5(\text{count}-1)^2$ | cost $C$ |
|---|---|---|---|
| (0, 0) | 0 | 5 | **5** |
| (1, 0) | $-3$ | 0 | **$-3$** |
| (0, 1) | $-2$ | 0 | **$-2$** |
| (1, 1) | $-5$ | 5 | **0** |

Expanding the square ($K=1$): $C=5+(-3-5)x_0+(-2-5)x_1+10\,x_0x_1$, so $c_0=5$, $l_0=-8$, $l_1=-7$, $q_{01}=10$. Equation (3): $c_0'=5+\frac{-8-7}2+\frac{10}4=0$, $h_0=4-\frac{10}4=1.5$, $h_1=3.5-2.5=1$, $J_{01}=2.5$. So $H=1.5\,Z_0+Z_1+2.5\,Z_0Z_1$. Check $(1,0)$: $Z_0=-1,Z_1=+1$: $-1.5+1-2.5=-3$ ✓. Check $(1,1)$: both $-1$: $-1.5-1+2.5=0$ ✓.

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**Bit order.** Variable $x_i$ is qubit $i$, and layout $x$ sits at position $k=\sum_i x_i2^i$ of the list of 64 numbers. Qiskit prints strings with qubit 0 on the **right**: the string `'001101'` means $x=(1,0,1,1,0,0)$. Never index count strings by hand; use `qff.bits.to_x` and `index_to_x`, as the cells below do.

</div>

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Hamiltonian | مؤثر هاملتون (الهاملتوني) | G | Pauli gates (X, Y, Z) | بوابات باولي | G |
| Operator | المؤثر | G | Ground state | الحالة الأرضية | I |
| Eigenvalue | القيمة الذاتية | G | Diagonal matrix | مصفوفة قطرية | I |
| Eigenstate | الحالة الذاتية | G | Matrix | مصفوفة | I |

In [ ]:
# Equation (3) in code, checked against Qiskit's to_ising(): first the tiny example, then the real problem.
def ising_by_formula(qp):
    """Equation (3): constant c0', Z coefficients h_i and ZZ coefficients J_ij of the Hamiltonian of a QUBO."""
    n = qp.get_num_vars()
    lin, pairs = np.zeros(n), {}
    for i, v in qp.objective.linear.to_dict().items():
        lin[i] += v
    for (i, j), v in qp.objective.quadratic.to_dict().items():
        if i == j:
            lin[i] += v                                   # x*x = x for a 0/1 variable
        else:
            pairs[tuple(sorted((i, j)))] = pairs.get(tuple(sorted((i, j))), 0) + v
    h = -lin / 2
    for (i, j), v in pairs.items():
        h[i] -= v / 4
        h[j] -= v / 4
    return qp.objective.constant + lin.sum() / 2 + sum(pairs.values()) / 4, h, {k: v / 4 for k, v in pairs.items()}


def check_ising(qubo, op, off):
    """Stop with an error if Qiskit's Hamiltonian (op, off) differs from equation (3)."""
    c0, h, J = ising_by_formula(qubo)
    assert abs(off - c0) < 1e-9, "constant differs"
    for pauli, coeff in zip(op.paulis, op.coeffs):
        on = [i for i in range(op.num_qubits) if pauli.z[i]]
        want = h[on[0]] if len(on) == 1 else J[tuple(on)]
        assert abs(coeff.real - want) < 1e-9, (pauli, coeff, want)


# The tiny example: two sites, yields 3 and 2, exactly one turbine, penalty 5.
qp_mini = QuadraticProgram("mini")
qp_mini.binary_var("x0")
qp_mini.binary_var("x1")
qp_mini.minimize(linear={"x0": -3, "x1": -2})
qp_mini.linear_constraint(linear={"x0": 1, "x1": 1}, sense="==", rhs=1, name="exactly_one")
qubo_mini = QuadraticProgramToQubo(penalty=5).convert(qp_mini)
op_mini, off_mini = qubo_mini.to_ising()
check_ising(qubo_mini, op_mini, off_mini)
diag_mini = qff.classical.diag_of(op_mini, off_mini)
print("tiny example: Hamiltonian", {k: float(v) for k, v in zip(op_mini.paulis.to_labels(), op_mini.coeffs.real)}, "| constant:", off_mini)   # Z0 1.5, Z1 1, Z0Z1 2.5
print("cost of (0,0), (1,0), (0,1), (1,1):", diag_mini.tolist())                                                        # [5, -3, -2, 0]
assert diag_mini.tolist() == [5.0, -3.0, -2.0, 0.0]

# The real problem.
check_ising(qubo, cost_op, offset)
print("OK: equation (3) matches Qiskit's Hamiltonian for both problems.")
print("real problem: h_i =", np.round(ising_by_formula(qubo)[1], 2), "| constant c0' =", ising_by_formula(qubo)[0])

### 3.3 · Why lowering the average energy finds the best layout

Prepare any quantum state $|\psi\rangle=\sum_x\alpha_x|x\rangle$ ($\alpha_x$ are the **amplitudes**: numbers that may be negative or complex, whose squared sizes are probabilities). Measuring gives layout $x$ with probability $p(x)=|\alpha_x|^2$, and the probabilities add up to 1. The **average energy** (the *expectation value*) of the state is the average cost you would see over many measurements:

$$\langle H\rangle=\sum_x p(x)\,C(x)\qquad\Bigl(=\langle\psi|H|\psi\rangle\Bigr) \tag{4}$$

Because $H$ is diagonal, the two forms are identical. Now the key fact. Let $C_{\min}$ be the lowest cost (the ground-state energy). Then

$$\langle H\rangle-C_{\min}=\sum_x p(x)\,\bigl(C(x)-C_{\min}\bigr)\ \ge\ 0 \tag{5}$$

since every probability is $\ge0$ and every bracket is $\ge0$. So **no state can have an average energy below $C_{\min}$**, and the average equals $C_{\min}$ only when *all* the probability sits on layouts of cost $C_{\min}$, the best ones. This is the **variational principle**. It turns a search into a descent: *the lower the average energy, the better the state; the very bottom is the best layout.* We never need to know the answer in advance, only to keep lowering $\langle H\rangle$.

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Expectation value (average energy) | القيمة المتوقعة (متوسط الطاقة) | I | Measurement | القياس | G |
| Variational principle | المبدأ التغايري | I | Quantum state | الحالة الكمومية | I |
| Amplitude | السعة | G | Variational | التغايري / المتغايرة | G |
| Probability | الاحتمال | I |  |  |  |

In [ ]:
# Equations (4) and (5) in code: the average energy of 3000 random states never goes below the lowest cost -28.
rng_states = np.random.default_rng(0)
random_states = rng_states.normal(size=(3000, 64))
random_states /= np.linalg.norm(random_states, axis=1, keepdims=True)          # 64 amplitudes, squares add up to 1
avg_energy = (random_states ** 2) @ diag                                 # equation (4): sum_x p(x) C(x) for each state
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(avg_energy, bins=40, color="#0F62FE")
ax.axvline(diag.min(), color="k", ls="--", label=f"lowest cost {diag.min():.0f}: no state can go below")
ax.set_xlabel("average energy <H> of a random state"); ax.set_ylabel("number of states"); ax.legend(fontsize=8)
print(f"lowest average energy among 3000 random states: {avg_energy.min():.2f}  (the floor is {diag.min():.0f})")
best_state = np.zeros(64); best_state[int(np.argmin(diag))] = 1.0               # all probability on the best layout
print("state sitting on the best layout: <H> =", best_state ** 2 @ diag, "= the floor")
assert avg_energy.min() >= diag.min() - 1e-9

<div style="border:2px dashed #007D79;border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🧪 Try it · جرّب</span>

Change `random_states` so that half of each state's squared weight sits on the best layout (hint: add `5 * best_state` before normalising). What happens to the histogram? Why can it still never reach the dashed line unless *all* the weight is on the best layout?

</div>

### 3.4 · The tunable circuit (the ansatz)

We cannot write down the best state; we have to *find* it. A quantum circuit with adjustable **knobs** (angles) prepares a whole family of states, one per setting of the knobs. Such a circuit is called an **ansatz** (a German word for "starting guess"). Ours is `real_amplitudes`: layers of **$R_y$ rotations**, joined by **CNOT** gates.

- **$R_y(\theta)$** rotates one qubit by the angle $\theta$: it turns $|0\rangle$ into $\cos\frac\theta2\,|0\rangle+\sin\frac\theta2\,|1\rangle$, so the probability of reading 1 is $\sin^2\frac\theta2$. $\theta=0$ keeps $|0\rangle$, $\theta=\pi$ gives $|1\rangle$. All amplitudes stay real numbers (hence the name).
- **CNOT** (controlled-NOT) flips the *target* qubit when the *control* qubit is 1. A chain of CNOTs between neighbouring qubits makes the qubits **entangled**: their probabilities are no longer independent. Without it a circuit could only describe each site on its own, but a good layout depends on *which sites go together* (wake losses are about pairs).

With one layer ($\texttt{reps}=1$) the circuit is: $R_y$ on every qubit, CNOTs in a line, a final $R_y$ on every qubit: $6+6=12$ knobs $\theta_1,\dots,\theta_{12}$, written $\boldsymbol\theta$. For each setting it prepares one state with amplitudes $\alpha_x(\boldsymbol\theta)$, and by (4) an average energy

$$\langle H\rangle(\boldsymbol\theta)=\sum_x\bigl|\alpha_x(\boldsymbol\theta)\bigr|^2\,C(x) \tag{6}$$

Only **one** thing changes when the optimiser turns a knob: the 64 probabilities. For a single qubit prepared by $R_y(\theta)$ and measured with $Z$ (eigenvalue $+1$ for $|0\rangle$, $-1$ for $|1\rangle$), the average is $\cos^2\frac\theta2-\sin^2\frac\theta2=\cos\theta$:

$$\langle Z\rangle=\cos\theta \tag{7}$$

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Ansatz | الافتراض المبدئي (ansatz) | G | Quantum circuit | الدائرة الكمومية | U |
| Rotation gate (Ry) | بوابة دوران | I | Knob (parameter) | المعامل (المقبض) | I |
| CNOT gate | بوابة النفي المتحكّم بها | G | Circuit depth | عمق الدائرة | G |
| Entanglement | التشابك | G | Quantum gate | البوابة الكمومية | U |

In [ ]:
# Step 3: the tunable circuit.
ansatz = real_amplitudes(6, reps=1, entanglement="linear")
print("number of knobs:", ansatz.num_parameters)
try:
    display(ansatz.draw("mpl", fold=40))
except Exception:                                      # drawing with matplotlib needs pylatexenc; fall back to text
    print(ansatz.draw("text", fold=120))

# Equation (7): <Z> = cos(theta) for one qubit prepared by Ry(theta); check it on a single-qubit circuit.
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
theta_demo = 0.9
one_qubit = QuantumCircuit(1)
one_qubit.ry(theta_demo, 0)
z_avg = Statevector(one_qubit).expectation_value(SparsePauliOp("Z")).real
assert abs(z_avg - np.cos(theta_demo)) < 1e-12
print(f"equation (7): <Z> = {z_avg:.4f} = cos({theta_demo})")

**A worked example you can check on paper: VQE on the tiny two-site problem.** Use the Hamiltonian from 3.2, $H=1.5\,Z_0+Z_1+2.5\,Z_0Z_1$, and the simplest ansatz: one $R_y$ on each qubit, no CNOT, knobs $(\theta_0,\theta_1)$. The two qubits are independent, so $\langle Z_0\rangle=\cos\theta_0$, $\langle Z_1\rangle=\cos\theta_1$ and $\langle Z_0Z_1\rangle=\cos\theta_0\cos\theta_1$. Equation (7) gives

$$\langle H\rangle(\theta_0,\theta_1)=1.5\cos\theta_0+\cos\theta_1+2.5\cos\theta_0\cos\theta_1 \tag{M}$$

| knobs $(\theta_0,\theta_1)$ | state | $\langle H\rangle$ by (M) |
|---|---|---|
| $(0,0)$ | layout $(0,0)$ | $1.5+1+2.5=5$ |
| $(\pi,0)$ | layout $(1,0)$ | $-1.5+1-2.5=-3$ ← the best layout |
| $(0,\pi)$ | layout $(0,1)$ | $1.5-1-2.5=-2$ ← the second best |
| $(\pi,\pi)$ | layout $(1,1)$ | $-1.5-1+2.5=0$ |

At knobs $0$ and $\pi$ the circuit prepares a single layout, so $\langle H\rangle$ is just that layout's cost: the four numbers of the table in 3.2. In between, (M) interpolates smoothly. Now notice something: near $(0,\pi)$ the surface curves **up in every direction** (the value $-2$ is lower than all its neighbours), but it is **not the lowest** value: $-3$ is lower. A point like that is a **local minimum**. An optimiser that only ever walks downhill and starts near $(0,\pi)$ stays there and reports the second-best layout. The next cell draws the whole surface and checks (M) against Qiskit.

<div style="border:1px solid rgba(0,125,121,0.35);border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">💡 Idea · فكرة</span>

**This is the whole drama of VQE in two knobs.** The best layout is the global minimum; the second-best layout is a local minimum that traps some starting points. With six qubits and 12 knobs there are many more traps, and that is why section 3.6 restarts the search from several random starting points.

</div>

In [ ]:
# The tiny VQE example: equation (M) against Qiskit's estimator, and the whole energy surface.
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

est_mini = StatevectorEstimator()
ansatz_mini = QuantumCircuit(2)
ansatz_mini.ry(Parameter("t0"), 0)
ansatz_mini.ry(Parameter("t1"), 1)                                                   # one Ry per qubit, no CNOT


def energy_mini(t0, t1):
    ev = est_mini.run([(ansatz_mini, op_mini, [[t0, t1]])]).result()[0].data.evs
    return float(np.ravel(ev)[0]) + off_mini                                         # add the constant back


def energy_by_hand(t0, t1):
    return 1.5 * np.cos(t0) + np.cos(t1) + 2.5 * np.cos(t0) * np.cos(t1)             # equation (M)


print(f"{'theta0':>8}{'theta1':>8}{'by hand (M)':>14}{'Qiskit':>10}")
for t0, t1 in [(0, 0), (np.pi, 0), (0, np.pi), (np.pi, np.pi), (1.0, 2.0)]:
    print(f"{t0:8.3f}{t1:8.3f}{energy_by_hand(t0, t1):14.4f}{energy_mini(t0, t1):10.4f}")
    assert abs(energy_by_hand(t0, t1) - energy_mini(t0, t1)) < 1e-9, "equation (M) disagrees with Qiskit"
qff.plots.plot_landscape(energy_by_hand, (0, 2 * np.pi), (0, 2 * np.pi), n=61, xlabel="theta_0", ylabel="theta_1",
                         title="average energy of the tiny example (dark = low = good)",
                         marks=[(np.pi, 0, "best: -3 (layout 10)", "source"), (0, np.pi, "local minimum: -2 (layout 01)", "use")]);

### 3.5 · Measuring the average energy

For one setting of the 12 knobs, the `StatevectorEstimator` computes $\langle H\rangle(\boldsymbol\theta)$ exactly from the state vector (equation 6; no shot noise, which is why a simulator is convenient for learning). On a real device the same number is estimated from many runs. Qiskit's Hamiltonian leaves out the constant `offset` (3.2), so the function below adds it back. Two settings to try: all knobs at zero (every qubit stays $|0\rangle$, the layout "no turbines", which breaks the rule and costs $0+12\cdot9=108$ in this notebook's units, as the diagonal shows) and a random setting. The cell then **checks** the estimator against equation (4) by computing $\sum_xp(x)C(x)$ from the 64 probabilities.

In [ ]:
estimator = StatevectorEstimator()


def energy_at(theta):
    """Average energy <H> + offset of the ansatz at knob values theta (the estimator itself leaves out the offset)."""
    ev = estimator.run([(ansatz, cost_op, [list(theta)])]).result()[0].data.evs
    return float(np.ravel(ev)[0]) + offset


theta_random = np.random.default_rng(1).uniform(-np.pi, np.pi, 12)
print("all knobs zero  : <H> =", round(energy_at(np.zeros(12)), 2), "(= the cost of the layout 000000, which is illegal)")
print("random knobs    : <H> =", round(energy_at(theta_random), 2))
print("lowest possible : ", diag.min())

In [ ]:
# Check equation (4): the estimator's answer equals sum_x p(x) C(x) computed from the 64 probabilities.
probs_random = Statevector(ansatz.assign_parameters(theta_random)).probabilities()
assert abs(energy_at(theta_random) - probs_random @ diag) < 1e-9, "estimator and sum p(x) C(x) should agree"
print("OK: <H> from the estimator =", round(energy_at(theta_random), 4), "= sum of p(x) * C(x) =", round(float(probs_random @ diag), 4))
print("floor (equation 5): no setting of the knobs can give less than", diag.min())

### 3.6 · The VQE loop, and why we restart it

The loop is a conversation between a quantum circuit and an ordinary program:

$$\boldsymbol\theta^*=\arg\min_{\boldsymbol\theta}\ \langle H\rangle(\boldsymbol\theta) \tag{8}$$

("argmin" means "the values at which the function is smallest".) The circuit (or here its simulator) turns a setting $\boldsymbol\theta$ into $\langle H\rangle$; a classical **optimiser** looks at the numbers it has seen and proposes a better $\boldsymbol\theta$; repeat until it stops improving. We use **COBYLA**, an optimiser that needs no derivatives: it only asks "what is $\langle H\rangle$ here?". When it stops, we **read the answer** off the final state: the layout with the highest probability.

As the tiny example showed, the landscape has valleys, so *where you start matters*. A **run** starts from random knob values (fixed by a **seed**, so everyone gets the same numbers), lets COBYLA lower $\langle H\rangle$ in at most 400 steps, then reads off the most likely layout. In T2 you write the run as a function and call it with five different seeds.

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Classical optimiser | المُحسِّن الكلاسيكي | I | Restart (random start) | إعادة التشغيل من نقطة بداية عشوائية | I |
| Local minimum | حدّ أدنى محلي | I | Seed | البذرة العشوائية | I |
| Hybrid quantum-classical loop | الحلقة الهجينة (كمومية وكلاسيكية) | I | VQE | محلّل القيم الذاتية الكمومي التغايري (VQE) | G |

In [ ]:
qff.plots.plot_flow([("Knobs", "theta", "source"), ("Ansatz", "equations 6-7", "store"), ("Average energy", "equation 4", "move"),
                     ("Optimiser", "COBYLA", "use")], loop=(3, 0, "new knob values"), title="one VQE run, repeated until <H> stops falling");

### T2 · Core step: one VQE run, then five restarts · التمرين ٢

**What you are about to build, and why.** This is the heart of the notebook. You wrap *one complete VQE run* in a function, `run_vqe(seed)`: build the ansatz, pick the random starting knobs from the seed, let the optimiser lower the average energy while a **callback** records every value it sees (so we can draw the descent), then read off the most likely layout and its energy. Then you call it for five seeds and keep the best run. The restarts are the lesson of the notebook.

**Two ways to do it.** The task cell already contains the plumbing (the ansatz, the `VQE` object with its callback, the run itself, the probabilities, the most likely layout, the timing and the five restarts) and leaves three blanks that carry the ideas: the reproducible random start, the energy you report (mind the offset), and the choice of the best run. Teams that want the full exercise can delete the scaffold and write everything from the Goal.

**The formulas you implement:** the start $\boldsymbol\theta_0$ is random in $[-\pi,\pi]$; the loop is equation (8); the energy you report is $\langle H\rangle+c_0'$ (the offset!); the layout you report is $x^*=\arg\max_x p_{\boldsymbol\theta^*}(x)$.

**Stuck? Open the hints one at a time, and try again after each. · تلميحات**

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 1 · a nudge · تلميح ١</summary>

Blank `x0`: which of the inputs of `VQE` must change from one restart to the next so that the five runs are genuinely different, and how do you make "random" repeatable for everybody? How many numbers does the start need? Blank `energy`: does the number the optimiser hands back already contain the whole energy of equation (4), or did section 3.2 split something off? Blank `best_run`: what do you compare, and do you want the smallest or the largest?

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 2 · the structure · تلميح ٢</summary>

**`x0`**: a reproducible random start. Make a generator seeded with `seed` (the Toolbox row "seeded random start" shows the call), draw one value per knob (the circuit tells you how many: `circuit.num_parameters`), uniformly between -pi and pi. **`energy`**: the eigenvalue that `compute_minimum_eigenvalue` returns, as a real number, is only the part of the energy without the constant offset of section 3.2; the energy you report must have it added back (the callback line above does the same for the history). Take care that you add it once, not twice. **`best_run`**: Python's `min` over `vqe_runs` with a `key` function that picks the `"energy"` entry of each run's dictionary.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 3 · nearly there · تلميح ٣</summary>

```python
x0 = np.random.default_rng(______).uniform(______, ______, circuit.num_parameters)
energy = result.eigenvalue.real + ______
best_run = min(vqe_runs, key=lambda r: r[______])
```

</details>

In [ ]:
# TODO (team) T2: write the VQE run as a function and restart it from seeds 1 to 5
# Goal: run_vqe(seed, op=cost_op, off=offset, reps=1) performs one complete VQE run and returns a dict with the keys
# seed, energy (offset included), x (the most likely layout), prob (its probability), history (energy at every step), circuit, theta.
# Then: vqe_runs = the five runs for seeds 1 to 5; best_run = the one with the lowest energy;
# best_vqe_energy, best_vqe_x = its energy and layout; vqe_seconds = the seconds the five runs took (time.perf_counter() before and after).
# EASIER ROUTE (default): the plumbing below is written for you. Fill the blanks marked ______ , then delete the "raise" line.
# CHALLENGE ROUTE: delete the scaffold and write everything yourself from the Goal above (the hints still apply).
def run_vqe(seed, op=cost_op, off=offset, reps=1):
    """One VQE run from the random start given by `seed`. Returns a dict: seed, energy (with offset), x (best layout), prob, history."""
    circuit = real_amplitudes(6, reps=reps, entanglement="linear")
    x0 = ______   # the reproducible random start made from `seed`: one value per knob, between -pi and pi (Toolbox: seeded random start)
    history = []
    vqe = VQE(StatevectorEstimator(), circuit, COBYLA(maxiter=400), initial_point=x0,
              callback=lambda n, params, value, meta: history.append(value + off))
    result = vqe.compute_minimum_eigenvalue(op)
    energy = ______    # the energy to report for this run (the eigenvalue is not the whole story: see the offset of section 3.2)
    probs = Statevector(circuit.assign_parameters(result.optimal_point)).probabilities_dict()
    top = max(probs, key=probs.get)
    return {"seed": seed, "energy": float(energy), "x": to_x(top), "prob": float(probs[top]), "history": history,
            "circuit": circuit, "theta": result.optimal_point}
raise NotImplementedError("T2: fill the three blanks ______ (x0, energy, best_run), then delete this line")


t0 = time.perf_counter()
vqe_runs = [run_vqe(seed) for seed in (1, 2, 3, 4, 5)]
vqe_seconds = time.perf_counter() - t0
best_run = ______    # the run with the lowest energy among vqe_runs
best_vqe_energy, best_vqe_x = best_run["energy"], best_run["x"]
print(f"five runs took {vqe_seconds:.1f} s")

In [ ]:
# Self-check
assert len(vqe_runs) == 5, "T2: run seeds 1, 2, 3, 4 and 5"
assert vqe_seconds > 0, "T2: time the five runs with time.perf_counter() and store the seconds in vqe_seconds (the results table uses it)"
assert all(len(r["history"]) > 20 for r in vqe_runs), "T2: record the energy at every evaluation with the callback"
assert min(r["energy"] for r in vqe_runs) >= classical_value - 1e-6, "T2: below the exact optimum is impossible; did you add the offset once?"
assert abs(best_vqe_energy - (-28)) < 0.5, "T2: the best of five runs should reach about -28"
assert best_vqe_x == classical_x, "T2: the best run should read out the same layout as brute force (use to_x on the count string)"
print("T2 passed")

In [ ]:
print(f"{'seed':<6}{'energy':<10}{'top layout (x0..x5)':<22}{'sites':<12}{'probability':<13}{'feasible':<10}{'objective'}")
for r in vqe_runs:
    sites = [i for i in range(6) if r["x"][i]]
    print(f"{r['seed']:<6}{r['energy']:<10.2f}{''.join(map(str, r['x'])):<22}{str(sites):<12}{r['prob']:<13.3f}{str(bool(qp.is_feasible(r['x']))):<10}{qp.objective.evaluate(r['x']):.0f}")
print(f"\nexact optimum: {classical_value:.0f} at {''.join(map(str, classical_x))}")
hits = sum(abs(r['energy'] - classical_value) < 0.5 for r in vqe_runs)
print(f"runs that reached the optimum: {hits} of 5")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
qff.plots.plot_convergence({f"seed {r['seed']}": r["history"] for r in vqe_runs}, exact=classical_value, ax=axes[0])
axes[0].set_ylabel("energy (with offset)"); axes[0].set_title("five VQE runs: five starting points")
draw_farm([i for i in range(6) if best_vqe_x[i]], f"best VQE layout (seed {best_run['seed']})", ax=axes[1])
plt.tight_layout()

**Local minima are the lesson, not a bug.** Four of the five runs stop in a valley that is *not* the lowest one: energies around −25 to −26, which are legal three-turbine layouts that are slightly worse than the best one. One run in five finds the best layout. We did not tune the settings until every seed worked, because that is what you would face on a problem where you do not know the answer. In practice you restart many times and keep the best, which multiplies the number of circuit evaluations.

**Before and after.** Where does the probability go? The first panel shows how the *final state of each of the five runs* spreads over the layouts, grouped by cost; the grey bars are random guessing. The run that reached the optimum puts nearly all its weight at cost -28; the others keep their weight on slightly worse legal layouts (cost -26 or -25). The second panel is the same information for the best run only, as a bar chart of its ten likeliest layouts.

In [ ]:
dists = {"random guessing": np.full(64, 1 / 64)}
for r in vqe_runs:
    dists[f"VQE seed {r['seed']}"] = Statevector(r["circuit"].assign_parameters(r["theta"])).probabilities()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
costs_sorted = np.unique(np.round(diag, 6))
low = costs_sorted[costs_sorted <= -20]                                         # zoom on the interesting end
for ax_ in axes[:1]:
    width = 0.8 / len(dists)
    palette = ["#8D8D8D", "#78A9FF", "#4589FF", "#0F62FE", "#0043CE", "#001D6C"]
    for k, (name, pr) in enumerate(dists.items()):
        vals = [pr[np.isclose(diag, c)].sum() for c in low]
        ax_.bar(np.arange(len(low)) + (k - (len(dists) - 1) / 2) * width, vals, width, label=name, color=palette[k])
    ax_.set_xticks(range(len(low))); ax_.set_xticklabels([f"{c:g}" for c in low], fontsize=8)
    ax_.set_xlabel("cost of the layout (only the cheap end: -20 and below)"); ax_.set_ylabel("probability"); ax_.legend(fontsize=7)
probs_best = dists[f"VQE seed {best_run['seed']}"]
top10 = np.argsort(probs_best)[::-1][:10]
qff.plots.plot_counts({"".join(map(str, index_to_x(int(k), 6))): float(probs_best[k]) for k in top10},
                      highlight=["".join(map(str, classical_x))], title=f"best run (seed {best_run['seed']}): ten likeliest layouts (optimum in red)", ax=axes[1])
plt.tight_layout()

<div style="border:2px dashed #007D79;border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🧪 Try it · جرّب</span>

**Count your own restarts.** In T2 you used the seeds 1 to 5. Call `run_vqe(s)` for the seeds 6 to 15 as well (each run takes a few seconds) and count how many of the fifteen runs end at the best energy. Does the fraction "about one in five" hold up? If one run succeeds with probability $q$, then $m$ independent restarts all fail with probability $(1-q)^m$: how many restarts would you need for a 90% chance of at least one success?

</div>

**A deeper circuit.** More layers means more knobs and a bigger family of states. Below, the same five seeds with `reps=2` (18 knobs). Deeper circuits can express better states, but do not remove the restart problem.

In [ ]:
t0 = time.perf_counter()
vqe_runs_deep = [run_vqe(seed, reps=2) for seed in (1, 2, 3, 4, 5)]
deep_seconds = time.perf_counter() - t0
print(f"{'seed':<6}{'energy (reps=2)':<18}{'top layout':<14}{'probability'}")
for r in vqe_runs_deep:
    print(f"{r['seed']:<6}{r['energy']:<18.2f}{''.join(map(str, r['x'])):<14}{r['prob']:.2f}")
print(f"runs whose top layout is the optimum {''.join(map(str, classical_x))}: {sum(r['x'] == classical_x for r in vqe_runs_deep)} of 5 | {deep_seconds:.1f} s for five runs")
print("(seed 2 ends at -27.02: its most likely layout is the optimum, but the state still carries weight on worse layouts)")

**Why does this work, and when does it not?** Equation (5) is the guarantee: lowering $\langle H\rangle$ can only move the state towards the best layouts, and the bottom of the valley is the best layout. What nothing guarantees is that the optimiser *reaches* the bottom. In practice:

- **Local minima** (the tiny example): a run can settle in a valley that is not the lowest one. Here four of five runs ended in a valley around $-26$; one in five found the best layout. In practice people restart many times and keep the best, which multiplies the number of circuit evaluations.
- For larger circuits the landscape can become almost flat (**barren plateaus**), and the optimiser has nothing to follow.
- A real device adds **noise**; every one of the hundreds of circuit evaluations is a little wrong, and the optimiser sees those errors.

In principle VQE can find the exact ground state if the ansatz is rich enough and the optimiser finds the minimum; neither is guaranteed at a useful size.

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**This is a toy.** 64 layouts are checked exhaustively in a millisecond. Nothing here shows that VQE beats a classical solver. See section 4.

</div>

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Local minimum | حدّ أدنى محلي | I | Quantum advantage | الأفضلية الكمومية | G |
| Barren plateau | الهضبة العقيمة | G | Ground state | الحالة الأرضية | I |
| Noise | الضوضاء | G |  |  |  |

### T3 · Stretch: what the penalty does · التمرين ٣

**What you are about to build, and why.** Equation (2) has one number you chose, the penalty $P$. T3 shows that this choice can quietly change *which problem you are solving*. You rebuild the Hamiltonian with $P=2$ and with $P=12$, find the lowest-energy layout of each **exactly** (the minimum of the diagonal), ask whether that layout obeys the rule, run VQE on both, and explain what you see.

**What to compute.** For each $P$: the diagonal $C_P(x)=f(x)+P(\sum_ix_i-K)^2$ (equation 2), its minimum and where it is attained, whether that layout has exactly $K$ turbines, and what the VQE restarts return.

**Stuck? Open the hints one at a time, and try again after each. · تلميحات**

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 1 · a nudge · تلميح ١</summary>

Think about equation (2) for a layout with *four* turbines when P = 2: what penalty does the fourth turbine pay, and what does it gain? Is breaking the rule now cheaper than obeying it? Then: how would you check that in code? The lowest-energy layout is where the diagonal of the Hamiltonian is smallest. Also decide what "obeys the rule" means in numbers, and which object in this notebook can answer yes or no for a given layout.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 2 · the structure · تلميح ٢</summary>

A loop over P = 2 and P = 12. For each: convert the problem with the penalty P and get the Hamiltonian and its offset; take the diagonal and find the position of its minimum, then turn that position into a layout; ask the problem whether that layout is feasible; run `run_vqe` for the five seeds on this Hamiltonian and keep the best by energy; store everything in `penalty_study[P]` under the six keys (`runs_at_lowest` is the number of the five runs whose final energy is within 0.5 of the lowest energy of this Hamiltonian). Finally write the explanation. Look at the printed numbers before you write: your sentence should name three facts, namely which layout is lowest for each P, whether it is legal, and how many of the five restarts reached it.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 3 · nearly there · تلميح ٣</summary>

```python
penalty_study = {}
for P in (2, 12):
    op_P, off_P = ______        # the Hamiltonian and offset for this penalty
    diag_P = ______             # its diagonal (cost of every layout)
    low_x = ______              # the lowest-energy layout of this Hamiltonian
    runs_P = ______             # five VQE runs on this Hamiltonian
    best_P = ______             # the best of them
    penalty_study[P] = {}       # lowest_energy, lowest_x, lowest_feasible, vqe_best_x, vqe_best_feasible, runs_at_lowest (how many of the five runs end within 0.5 of the lowest energy)
explanation = "..."             # what does the penalty do, and what goes wrong when it is too small?
```

</details>

In [ ]:
# TODO (team) T3: stretch: set the penalty to 2, compare with 12, and explain what the penalty does
# Goal: for P in (2, 12) build the penalised Hamiltonian, find its exact lowest-energy layout and whether that layout obeys the rule,
# run VQE (five seeds) on it, and fill penalty_study[P] = {"lowest_energy", "lowest_x", "lowest_feasible", "vqe_best_x",
# "vqe_best_feasible", "runs_at_lowest"}. Then write explanation = "...": what does the penalty do, and what goes wrong when it is too small?
penalty_study = {}
raise NotImplementedError("T3: rerun with penalty 2 and penalty 12, report what the classical and VQE answers do, explain")

In [ ]:
# Self-check
assert set(penalty_study) == {2, 12}, "T3: study penalty 2 and penalty 12"
assert penalty_study[12]["lowest_feasible"] and penalty_study[12]["lowest_energy"] == -28, "T3: with penalty 12 the lowest energy is the legal optimum, -28"
assert not penalty_study[2]["lowest_feasible"], "T3: with penalty 2 the lowest-energy layout should break the 'exactly 3' rule"
assert isinstance(explanation, str) and len(explanation) > 60, "T3: write your explanation in a sentence or two"

In [ ]:
circuit = best_run['circuit'].assign_parameters(best_run['theta'])        # the main circuit of this notebook: the best VQE run

### Optional: how this would go to a real IBM Quantum computer
This is a sketch and has not been checked against the current IBM documentation. Nothing here runs in this notebook.

```python
# Not run here. Needs: pip install qiskit-ibm-runtime, and an IBM Quantum account saved on your own machine.
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# service = QiskitRuntimeService()                      # reads credentials saved earlier; never paste a token in a notebook
# backend = service.least_busy(operational=True, simulator=False)
# isa = generate_preset_pass_manager(optimization_level=3, backend=backend).run(circuit)
# job = Sampler(mode=backend).run([isa], shots=4096)
```

The next cell does the offline part: it transpiles your circuit for a *pretend* device on this computer and reports its size. Nothing is sent to IBM Quantum.

In [ ]:
preview = qff.quantum.device_preview(circuit)      # `circuit` = the main circuit of this notebook; runs locally
print(f"depth {preview['depth']}, two-qubit gates {preview['two_qubit_gates']}")

<div style="border:1px solid rgba(208,38,112,0.35);border-left:6px solid #D02670;background:rgba(208,38,112,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#D02670;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🌍 Real world · في الواقع</span>

The preview above is the best VQE circuit on a pretend device. It is shallow, so a single run of it might come out reasonably on today's real hardware. But VQE does not run it once: it ran about 400 times per start, five starts, and every run is noisy, which blurs the energy the optimiser is trying to lower. That matters for our claim: the energies in this notebook are *ideal-simulator* numbers, not a promise about a real device.

</div>

**Glossary — المصطلحات.** Tier G = Quantabic headword, U = used consistently on Quantabic, I = proposed here and not yet checked against quantabic.com. Where Quantabic has a term, we use it.

| English | العربية | Tier |
|---|---|---|
| Qubit | كيوبت | G |
| Superposition | التراكب | G |
| Entanglement | التشابك | G |
| Measurement | القياس | G |
| Amplitude | السعة | G |
| Phase | الطور | U |
| Quantum circuit | الدائرة الكمومية | U |
| Quantum gate | البوابة الكمومية | U |
| Hadamard gate | بوابة هادامارد | G |
| Quantum algorithm | خوارزمية كمومية | U |
| Quantum simulator | المحاكي الكمومي | I |
| Classical bit | البت الكلاسيكي | I |
| Quantum speedup (quadratic / exponential) | تسريع تربيعي / أسي | U |
| VQE | محلّل القيم الذاتية الكمومي التغايري (VQE) | G |
| ansatz | الافتراض المبدئي (ansatz) | G |
| Hamiltonian | مؤثر هاملتون (الهاملتوني) | G |
| ground state | الحالة الأرضية | I |
| cost function | دالة التكلفة | I |
| optimization | التحسين (الأمثَلة) | I |
| expectation value | القيمة المتوقعة | I |
| penalty | العقوبة (penalty) | I |

## 4 · Compare — المقارنة

In [ ]:
# Results table: classical baseline rows first.
rows = [("brute force (64 layouts)", "".join(map(str, classical_x)), classical_value, brute_seconds, "64 layouts evaluated"),
        ("exact solver (NumPy eigensolver)", "".join(str(int(v)) for v in numpy_result.x), numpy_result.fval, numpy_seconds, "one matrix diagonalisation"),
        ("VQE, best of 5 starts (reps=1)", "".join(map(str, best_vqe_x)), best_vqe_energy, vqe_seconds,
         f"{sum(len(r['history']) for r in vqe_runs)} circuit evaluations"),
        ("VQE, mean of 5 starts (reps=1)", "-", float(np.mean([r['energy'] for r in vqe_runs])), vqe_seconds / 5, f"{len(vqe_runs[0]['history'])} per start")]
print(f"{'method':<36}{'layout x0..x5':<16}{'value':<9}{'seconds':<10}{'work'}")
for name, lay, val, secs, work in rows:
    print(f"{name:<36}{lay:<16}{val:<9.2f}{secs:<10.3f}{work}")
print("(for VQE the value is the energy with the offset added back; for the layout shown it equals the objective)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
labels = [f"seed {r['seed']}" for r in vqe_runs]
colors = ["#24A148" if abs(r["energy"] - classical_value) < 0.5 else "#8D8D8D" for r in vqe_runs]
axes[0].bar(labels, [r["energy"] for r in vqe_runs], color=colors)
axes[0].axhline(classical_value, ls="--", color="k", label="exact optimum (-28)")
axes[0].set_ylim(-30, -20); axes[0].set_ylabel("final energy"); axes[0].legend(loc="lower right")
axes[0].set_title("five starts: one reaches the optimum (green)")
draw_farm([i for i in range(6) if best_vqe_x[i]], "layout chosen by the best VQE run", ax=axes[1])
plt.tight_layout()

**What this toy shows.** A siting decision with interactions becomes an energy function. VQE lowers that energy with a parameterised circuit and, with restarts, lands on the same layout as exhaustive search (sites 0, 2 and 3, objective −28). One start in five got there; the other four stopped at −25 to −26.

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**What this toy does not show.** That a quantum computer is useful here. The 64 layouts are checked exhaustively in a millisecond; VQE needed hundreds of circuit evaluations per start and still failed from most starting points. Real farms have hundreds of sites, continuous positions and wake physics; nobody has shown VQE beating classical layout optimisers there.

</div>

**Paper versus our route.** Same route as the paper. WEF/Aramco Case 8 (p. 20) describes a QUBO formulation solved with hybrid VQE on IBM's Qiskit gate-based simulator, viable "for small instances". Our numbers (yields, wake penalties) are invented for teaching, so our results illustrate the *method*; they are not a reproduction of the paper's experiment.

In [ ]:
# Acceptance: the numbers this notebook promises
assert QUBITS == 6 and cost_op.num_qubits == 6
assert classical_x == [1, 0, 1, 1, 0, 0] and classical_value == -28
assert len(vqe_runs) == 5
assert best_vqe_x == classical_x and abs(best_vqe_energy - (-28)) < 0.5
assert sum(abs(r["energy"] + 28) < 0.5 for r in vqe_runs) < 5      # at least one run got stuck: the lesson holds
print("VQE energies:", [round(r["energy"], 2) for r in vqe_runs], "| exact:", classical_value)

## 5 · Build the explainer — ابنوا المنتَج
How you are scored depends on how you take part. Read your category below.

### A) In-person attendees: group effort
Your team builds **one product** together that teaches a visitor your use case and your algorithm. It is **50% of your score**. This category is a **group effort**: the whole team plans, builds, tests and presents the product together. The product does not have to be in Arabic.

**Your audience:** a first-year university student who has never seen a qubit. They have three minutes.

**Your product must answer four questions, in this order:**
1. What is the problem, and who has it? Choose which sites get a wind turbine so that total yield minus wake losses is as good as possible.
2. Why is it hard when it gets big? 6 sites give 64 layouts to check; 60 sites give about 10¹⁸. Smart classical methods do not check them all, so the honest question is where they stop scaling.
3. What does the quantum algorithm do differently? Turn "good layout" into "low energy" and let a tunable circuit roll downhill; since it can get stuck, restart it from several places.
4. How real is it today? (Use your section 4 results and the honesty statement.)

**Pick any format, except a PowerPoint presentation (.pptx).** It can be an interactive demo, an interactive website, a YouTube video, a small app, a game, or an interactive notebook, as long as it does not take more than 5 minutes.

**Three ideas that fit this stop** (take one, change it, or ignore them):
1. Drag-and-drop turbines with a live yield score next to VQE's pick
2. A "valleys" animation of five starts ending in different places
3. A short film from the wind's point of view

**Use from this notebook:** the farm picture with the optimum (`draw_farm`), the five-restart table and convergence plot (energies about −25.9, −25.3, −28.0, −26.0, −25.5; only one reaches −28)

Qiskit integration and code is a separate 10%: keep your completed notebook tidy and show one cell of it.

**Time plan for the 2 h 20 build window:** 20 min agree the story and format · 70 min build · 20 min test on someone outside the team · 30 min rehearse the pitch.

### B) Online attendees: coding
The coding is worth **100 points**. You work **individually** and solve **one challenge**.

## 6 · Pitch — العرض
You have **5 minutes** for your pitch and 3 minutes for questions from the judges. Presentation is 40% of your score.

| Minutes | What | Who |
|---|---|---|
| 0–0.5 | Where we are in the journey: stop, use case, one sentence | 1 speaker |
| 0.5–1.5 | The problem and why it is hard at scale | |
| 1.5–3 | The algorithm, with your toy example and one notebook cell | |
| 3–4 | Live demo of your product | |
| 4–5 | How real is it: your result against the classical baseline | |
| 5-8 | Questions | all |

**Questions you should expect**
1. Did the quantum method beat the classical one? (Answer honestly: it matched brute force on 6 sites after restarts; it was slower.)
2. What happens when the problem is 100 times bigger?
3. Why this algorithm and not another?
4. Why did four of five runs get the wrong layout?
5. How did you choose the penalty?

**Checklist before you walk up**
- [ ] The notebook runs top to bottom and the self-checks pass
- [ ] Every number on screen comes from your notebook or from a named source
- [ ] You say what the toy does not show
- [ ] You finish the talk in 5 minutes on a timer

**Arabic is optional.** If you want to build or present your product in Arabic, use Quantabic for the terms: start with the glossary cell at the end of section 3, which follows Quantabic's terms. If a term is not in the glossary, check quantabic.com, and if it is not there either, give the English in brackets the first time. Keep gate letters, kets and symbols in Latin (H, |0⟩, θ). The words are yours: mentors can check your physics, but they will not write your Arabic.